# Notebook 1: What Does "Good" Mean?

Before you can measure quality, you have to define it. This notebook builds a **golden dataset** — a small set of questions with known-correct answers — and introduces the three failure modes that evaluations need to catch.

## Setup

In [1]:
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
client = OpenAI()
MODEL           = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
DATA_PATH       = Path("../data/AI_Agent_Insure.md")

## Part 1: The four failure Modes

A RAG or agent system can fail in four distinct places. Each requires a different evaluation strategy.

| Failure mode | What goes wrong | Where to measure |
|---|---|---|
| **Bad retrieval** | Wrong chunks returned — the answer isn't in the context | Retrieval layer (Notebook 2) |
| **Bad generation** | Right chunks, wrong answer — model ignores or misreads context | Generation layer (Notebook 3) |
| **Bad grounding** | Answer not supported by the retrieved context — model hallucinates | Generation layer (Notebook 3) |
| **Bad routing** | Agent calls the wrong tool or passes wrong arguments | Agent layer (Notebook 4) |

Measuring the system end-to-end only tells you *something* is wrong. Measuring each layer separately tells you *where* to fix it.

## Part 2: Build the RAG index

We build the same index used in Modules 4–6 so the golden dataset is grounded in the actual document.

In [2]:
def load_and_chunk(path: Path) -> list[str]:
    doc = path.read_text(encoding="utf-8").strip()
    chunks = [s.strip() for s in doc.replace("\n", " ").split(".") if s.strip()]
    return [c + "." for c in chunks]


chunks = load_and_chunk(DATA_PATH)
print(f"Loaded {len(chunks)} chunks")

resp       = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks)
embeddings = [r.embedding for r in resp.data]

chroma = chromadb.EphemeralClient()
coll   = chroma.create_collection("evals_rag")
coll.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks
)
print("Index ready.")

Loaded 34 chunks
Index ready.


## Part 3: The golden dataset

A golden dataset is a set of `(question, expected_answer, relevant_chunk_ids)` tuples you create by hand.

**Why by hand?** Because you need ground truth that isn't generated by the same model you're evaluating. If you ask the model to write its own test cases, you're measuring self-consistency, not correctness.

Each entry has:
- `question` — what a user would actually ask
- `expected_answer` — a reference answer you've verified against the document
- `relevant_chunk_ids` — which chunk indices *should* be retrieved for this question
- `expected_tools` — which tools an agent *should* call (used in Notebook 4)

The dataset lives in `data/golden_dataset.json` — edit it directly. Run the chunk printout below to identify the correct `relevant_chunk_ids` for each question, then add them to the JSON file.

In [3]:
# Print the chunks with their IDs so we can identify which ones are relevant
# for each golden question below.
print(f"{'ID':>4}  Chunk")
print("-" * 80)
for i, chunk in enumerate(chunks):
    print(f"{i:>4}  {chunk[:100]}")

  ID  Chunk
--------------------------------------------------------------------------------
   0  # AI Agent Insure ## Comprehensive Company Profile & Operational Overview  This document provides a 
   1  The document is structured for ingestion into AI retrieval systems, including vector databases and R
   2  ---  # 1.
   3  Company Overview  AI Agent Insure is a specialty insurance organization focused exclusively on risks
   4  The company is designed as an AI-native insurer rather than a traditional cyber or technology E&O pr
   5  AI Agent Insure provides tailored insurance solutions addressing operational, regulatory, financial,
   6  Mission & Vision  ## Mission  To enable safe and scalable AI innovation by providing specialized ris
   7  ## Vision  To become the global standard insurer for agentic AI and autonomous systems, recognized f
   8  ---  # 3.
   9  Core Services  AI Agent Insure delivers services across underwriting, policy issuance, risk evaluati
  10  ### Services 

In [4]:
# Load the golden dataset from the JSON file.
# To update the dataset — add questions, fill in relevant_chunk_ids, correct an answer —
# edit data/golden_dataset.json directly and re-run this cell.
GOLDEN_PATH    = Path("../data/golden_dataset.json")
GOLDEN_DATASET = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

print(f"Loaded {len(GOLDEN_DATASET)} questions from {GOLDEN_PATH}")
for entry in GOLDEN_DATASET:
    print(f"  [{entry['id']}] ({entry['category']:12}) {entry['question']}")

Loaded 10 questions from ../data/golden_dataset.json
  [q01] (product_info) What does Agentic AI Liability Insurance cover?
  [q02] (pricing     ) How much would Model & Data Security Insurance cost for a startup?
  [q03] (eligibility ) Can a healthcare company get coverage from AI Agent Insure?
  [q04] (rag         ) How does AI Agent Insure handle claims?
  [q05] (rag         ) What is AI Agent Insure's underwriting philosophy?
  [q06] (product_info) What does Autonomous Systems & Robotics Coverage include?
  [q07] (pricing     ) How much does Agentic AI Liability Insurance cost for an enterprise?
  [q08] (eligibility ) Is a synthetic data company eligible for coverage?
  [q09] (multi_tool  ) I'm a robotics startup. Am I eligible and what would coverage cost?
  [q10] (multi_tool  ) Explain the claims process and what Model & Data Security Insurance costs for a mid-market company.


## Part 4: What makes a good golden dataset?

Ten questions is enough to start. In practice you want:

- **Coverage across failure modes** — questions that test retrieval, generation, and routing separately
- **Coverage across question types** — simple lookups, multi-step reasoning, edge cases
- **Verified answers** — read the source document and confirm each expected answer is correct
- **Realistic questions** — written the way a real user would ask, not the way the document is structured

What you want to avoid:
- Questions where the expected answer is a verbatim copy of a chunk (tests memorisation, not reasoning)
- Questions with ambiguous correct answers (makes scoring unreliable)
- Questions that are too easy (every system gets them right — no signal)

## Part 5: Inspect the dataset by category

In [5]:
from collections import Counter

counts = Counter(e["category"] for e in GOLDEN_DATASET)
print("Questions by category:")
for cat, n in counts.items():
    print(f"  {cat:15} {n}")

print()
print("Questions requiring multiple tools:")
for e in GOLDEN_DATASET:
    if len(e["expected_tools"]) > 1:
        print(f"  [{e['id']}] tools={e['expected_tools']}")
        print(f"         Q: {e['question']}")

Questions by category:
  product_info    2
  pricing         2
  eligibility     2
  rag             2
  multi_tool      2

Questions requiring multiple tools:
  [q09] tools=['check_eligibility', 'get_pricing_estimate']
         Q: I'm a robotics startup. Am I eligible and what would coverage cost?
  [q10] tools=['search_docs', 'get_pricing_estimate']
         Q: Explain the claims process and what Model & Data Security Insurance costs for a mid-market company.


## Key concepts

- Evaluations require ground truth you created, not generated by the model under test
- There are four distinct failure modes — each needs a different measurement strategy
- A golden dataset is small (10–50 questions) but carefully chosen
- The dataset lives in `data/golden_dataset.json` — edit it there, not in a notebook
- **Next:** Notebook 2 uses this dataset to evaluate the retrieval layer in isolation